In [2]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

Movies Shape: (9742, 3)
Ratings Shape: (100836, 4)


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
movies.isnull().sum()
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [6]:
print("Total Movies:", movies.shape[0])
print("Total Ratings:", ratings.shape[0])
print("Unique Users:", ratings["userId"].nunique())
print("Unique Movies Rated:", ratings["movieId"].nunique())

Total Movies: 9742
Total Ratings: 100836
Unique Users: 610
Unique Movies Rated: 9724


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
movies["genres"] = movies["genres"].fillna("")

cv = CountVectorizer()

genre_matrix = cv.fit_transform(movies["genres"])

print(genre_matrix.shape)

(9742, 24)


In [10]:
print(genre_matrix.shape)

(9742, 24)


In [11]:
similarity = cosine_similarity(genre_matrix)

print(similarity.shape)

(9742, 9742)


In [14]:
#recommend fnc
def recommend(movie_name):

    movie_index = movies[movies["title"] == movie_name].index[0]

    similarity_scores = list(
        enumerate(similarity[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nMovies similar to '{movie_name}':\n")

    for i, movie in enumerate(similarity_scores[1:6], start=1):

        movie_data = movies_with_ratings.iloc[movie[0]]

        print(
            f"{i}. {movie_data['title']} | "
            f"Rating: {round(movie_data['avg_rating'],2)}"
        )

In [20]:
recommend("Toy Story (1995)")


Movies similar to 'Toy Story (1995)':

1. Antz (1998)
2. Toy Story 2 (1999)
3. Adventures of Rocky and Bullwinkle, The (2000)
4. Emperor's New Groove, The (2000)
5. Monsters, Inc. (2001)


In [16]:
movie_ratings = ratings.groupby("movieId")["rating"].mean().reset_index()

movie_ratings.rename(
    columns={"rating": "avg_rating"},
    inplace=True
)

movie_ratings.head()

,movieId,avg_rating
0,1,3.920930
1,2,3.431818
2,3,3.259615
3,4,2.357143
4,5,3.071429


In [17]:
movies_with_ratings = movies.merge(
    movie_ratings,
    on="movieId",
    how="left"
)

movies_with_ratings.head()

,movieId,title,genres,avg_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143
4,5,Father of the Bride Part II (1995),Comedy,3.071429


In [18]:
movies_with_ratings.sort_values(
    by="avg_rating",
    ascending=False
)[["title", "avg_rating"]].head(10)

,title,avg_rating
9711,Won't You Be My Neighbor? (2018),5.0
4675,Jane Eyre (1944),5.0
3807,Rain (2001),5.0
7945,Goodbye Charlie (1964),5.0
2938,Sorority House Massacre (1986),5.0
2937,Slumber Party Massacre III (1990),5.0
2936,Slumber Party Massacre II (1987),5.0
5025,True Stories (1986),5.0
9367,Moonlight,5.0
9365,Tom Segura: Mostly Stories (2016),5.0
